# T04 · MCTS with the trained A2C network

We take the FC128 (Kartal) network trained in T03 and use its policy and value to guide a tree search
(`AlphaZeroAgent`, Gumbel MuZero from `mctx`). The main setting is the search budget, `num_simulations`.
With `0` simulations the agent simply plays the network, which is exactly the A2C agent.

We compare two networks: the A2C network as it is, and the same network after a short AlphaZero self-play
stage (`alphazero.train`, starting from the A2C weights), which trains the network towards the moves the
search chose and the returns it observed.

Every budget plays two match-ups: 1 vs. 1 against the rule-based agent (our standard evaluation) and against
the plain A2C network. We also measure how long a move takes for each budget, since the competition allows
0.5 s per move.

The search uses the reward of T03's last stage (`REWARD`). The critic was trained on this reward, so the
rewards along a path in the tree and the network's value at its end are on the same scale.

**Needs** `runs/a2c_17x17/FC128/versus.eqx` from T03. **Writes** `runs/mcts/`: `eval.csv` (one row per game),
`latency.csv`, `history.csv` (self-play stage), `selfplay/network.eqx` and `config.json`. **Used by** A07.

In [ ]:
import os

os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", os.path.expanduser("~/.cache/jax"))
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")

import json
import pathlib
import time

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

from bomberman.agents import A2CObjective, PolicyAgent, RuleBasedAgent
from bomberman.agents.alphazero import AlphaZeroAgent, Config, train
from bomberman.agents.networks import KartalCNNActorCriticNetwork
from bomberman.game import Game
from bomberman.training import (
    CoinPickupReward,
    DeathPenalty,
    KillReward,
    PBRSReward,
    PrintLogger,
    Rollout,
    StepPenalty,
    UrgentCrateReward,
    versus_task_settings,
)
from bomberman.training.scenarios import NoReward

HERE = pathlib.Path.cwd().resolve()  # run from the notebook's folder or from the repository root
EXP = next(p / "experiments" for p in (HERE, *HERE.parents) if (p / "experiments" / "README.md").exists())
RUNS = EXP / "runs"
OUT = RUNS / "mcts"
OUT.mkdir(parents=True, exist_ok=True)
print(f"backend: {jax.default_backend()} | output: {OUT}")

## Configuration

In [ ]:
SIZE = 17  # the board of T03
WEIGHTS = RUNS / "a2c_17x17" / "FC128" / "versus.eqx"
EVAL_SETTINGS = versus_task_settings(size=SIZE, crate_density=0.35, coins=9, max_steps=200, agents=2)
EVAL_GAME = Game(settings=EVAL_SETTINGS)
# reward of T03's last stage, which the critic learned to predict
REWARD = (CoinPickupReward() + KillReward() + DeathPenalty(death=3.0) + StepPenalty(step=0.02)
          + UrgentCrateReward(urgent_crate_scale=1.0) + PBRSReward(EVAL_SETTINGS))

BUDGETS = (0, 2, 4, 8, 16, 32)  # num_simulations; 0 means the plain network (the A2C agent)
EPISODES, CHUNK, LATENCY_MOVES = 128, 32, 64

# Self-play stage: the board of T03's last stage, a small search budget and a small learning rate, because
# the network already plays well and only needs fine-tuning.
TRAIN_SETTINGS = versus_task_settings(size=SIZE, coins=6, crate_density=0.18, max_steps=200, agents=2)
TRAIN_REWARD = (CoinPickupReward() + KillReward() + DeathPenalty(death=3.0) + StepPenalty(step=0.02)
                + UrgentCrateReward(urgent_crate_scale=1.0) + PBRSReward(TRAIN_SETTINGS))
TRAINING = dict(iterations=50, num_simulations=16, num_envs=64, batch_size=1024, learning_rate=1e-4,
                explore_moves=40)

a2c_network = eqx.tree_deserialise_leaves(
    WEIGHTS, KartalCNNActorCriticNetwork.for_settings(EVAL_SETTINGS, jax.random.key(0)))
bare = PolicyAgent(a2c_network, A2CObjective())  # opponent in the second match-up: always the original A2C network


def mcts(network, num_simulations):
    return AlphaZeroAgent(network=network, game=EVAL_GAME, reward_fn=REWARD, num_simulations=num_simulations)

## Self-play stage

The agent plays against copies of itself, all using search. The network learns to predict the moves the search
chose and the returns that followed. The metrics are logged per self-play iteration.

In [ ]:
start = time.time()
learner = AlphaZeroAgent(network=a2c_network, game=Game(settings=TRAIN_SETTINGS), reward_fn=TRAIN_REWARD,
                         num_simulations=TRAINING["num_simulations"])
learner, history = train(
    learner, jax.random.key(4), TRAINING["iterations"],
    Config(num_envs=TRAINING["num_envs"], batch_size=TRAINING["batch_size"],
           learning_rate=TRAINING["learning_rate"], explore_moves=TRAINING["explore_moves"]),
    logger=PrintLogger(every=5, keys=("loss", "policy_loss", "value_loss", "root_value", "coins", "death")))
print(f"{TRAINING['iterations']} iterations in {(time.time() - start) / 60:.1f} min")

selfplay_network = learner.network
(OUT / "selfplay").mkdir(exist_ok=True)
eqx.tree_serialise_leaves(OUT / "selfplay" / "network.eqx", selfplay_network)
history = pd.DataFrame(history).assign(iteration=lambda d: np.arange(len(d)))
history.to_csv(OUT / "history.csv", index=False)
history.tail()

## Evaluation

Every player picks its best action, and every configuration plays the same games (same random keys). The MCTS
agent is player 0. Its opponent is the rule-based agent or the plain A2C network, so with 0 simulations the A2C
network plays against itself.

In [ ]:
ROLLOUT = Rollout(EVAL_GAME, NoReward(), EVAL_SETTINGS.rules.max_steps)


def seat_stats(out, seat=0):
    """The result of every game for one player. A self-kill is a death in the player's own explosion."""
    states = out.states
    after = jax.tree.map(lambda a: a[:, 1:], states)
    died = states.alive[:, :-1, seat] & ~states.alive[:, 1:, seat]
    hit = jax.vmap(jax.vmap(lambda board: board.explosions.covers(board.agent_positions)))(after.board)
    score = states.score[:, -1]
    rival = jnp.delete(score, seat, axis=1, assume_unique_indices=True).max(axis=1)
    return dict(
        score=score[:, seat], rival_score=rival, win=score[:, seat] > rival, coins=out.coins[:, seat],
        death=~out.alive[:, seat], self_kill=(died & hit[:, :, seat, seat]).any(axis=1), steps=out.steps,
    )


@eqx.filter_jit
def play(player, opponent, key):
    return seat_stats(ROLLOUT([player, opponent], key, greedy=True, num_envs=CHUNK))


def evaluate(player, opponent, key):
    """Play `EPISODES` games, `CHUNK` at a time, and return one array per statistic."""
    chunks = [play(player, opponent, k) for k in jax.random.split(key, EPISODES // CHUNK)]
    return {k: np.concatenate([np.asarray(c[k], float) for c in chunks]) for k in chunks[0]}

In [ ]:
NETWORKS = {"A2C": a2c_network, "A2C + self-play": selfplay_network}
OPPONENTS = {"rule-based": RuleBasedAgent(), "A2C (bare network)": bare}
rows = []
for network_name, network in NETWORKS.items():
    for budget in BUDGETS:
        for opponent_name, opponent in OPPONENTS.items():
            start = time.time()
            stats = evaluate(mcts(network, budget), opponent, jax.random.key(1))
            rows += [dict(network=network_name, simulations=budget, opponent=opponent_name, episode=i,
                          **{k: v[i] for k, v in stats.items()}) for i in range(EPISODES)]
            print(f"{network_name:15s} {budget:3d} simulations vs {opponent_name:18s}: "
                  f"score {stats['score'].mean():.2f}  win {stats['win'].mean():.2f}  "
                  f"self-kill {stats['self_kill'].mean():.2f}  ({time.time() - start:.0f}s)")

evaluation = pd.DataFrame(rows)
evaluation.to_csv(OUT / "eval.csv", index=False)
evaluation.groupby(["network", "opponent", "simulations"])[["score", "rival_score", "win", "self_kill"]].mean().round(3)

## Decision latency

One move at a time, as in the competition: we time the compiled `greedy_action` on positions from real games.

In [ ]:
state = EVAL_GAME.init(jax.random.key(2))
samples = []  # (observation, legal actions) of player 0
for key in jax.random.split(jax.random.key(3), LATENCY_MOVES):
    seat = jnp.int32(0)
    samples.append((EVAL_GAME.observe(state, seat), EVAL_GAME.legal_action_mask(state, seat)))
    state = EVAL_GAME.step(state, jax.random.randint(key, (2,), 0, 5))  # random moves, no bombs

latency = []
for budget in BUDGETS:
    agent = mcts(a2c_network, budget)  # the self-play network has the same size, so it takes just as long
    decide = eqx.filter_jit(lambda agent, obs, mask, key: agent.greedy_action(obs, mask, key)[0])
    decide(agent, *samples[0], jax.random.key(0)).block_until_ready()  # compile
    for i, (obs, mask) in enumerate(samples):
        start = time.perf_counter()
        decide(agent, obs, mask, jax.random.key(i)).block_until_ready()
        latency.append(dict(simulations=budget, move=i, seconds=time.perf_counter() - start))

latency = pd.DataFrame(latency)
latency.to_csv(OUT / "latency.csv", index=False)
(OUT / "config.json").write_text(json.dumps(dict(
    size=SIZE, weights=str(WEIGHTS.relative_to(EXP)), network="FC128", budgets=BUDGETS, episodes=EPISODES,
    determinizations=1, discount=0.99, opponent_temperature=1.0, backend=jax.default_backend(),
    selfplay=dict(TRAINING, scenario=dict(coins=6, crate_density=0.18, max_steps=200, agents=2),
                  reward="versus-stage reward of T03"),
    eval=dict(crate_density=0.35, coins=9, max_steps=200, agents=2), tree_reward="versus-stage reward of T03",
), indent=2))
latency.groupby("simulations").seconds.describe(percentiles=[0.5, 0.95]).round(4)